# 03. Grounding 평가와 운영 Gate
목표: retrieval, citation coverage와 abstention을 분리해 평가합니다.

In [ ]:
cases=[
 {'id':'q1','gold_sources':{'d1'},'retrieved':['d1','d3'],'claims':2,'cited_claims':2,'should_answer':True,'answered':True},
 {'id':'q2','gold_sources':{'d2'},'retrieved':['d3'],'claims':1,'cited_claims':0,'should_answer':False,'answered':False},
 {'id':'q3','gold_sources':{'d3'},'retrieved':['d1','d3'],'claims':2,'cited_claims':1,'should_answer':True,'answered':True},
]
def evaluate(cases):
 recalls=[]; coverage=[]; abstention=[]
 for c in cases:
  recalls.append(len(c['gold_sources'] & set(c['retrieved']))/len(c['gold_sources']))
  coverage.append(c['cited_claims']/c['claims'] if c['claims'] else 1.0)
  abstention.append(c['answered']==c['should_answer'])
 return {'retrieval_recall':sum(recalls)/len(recalls),'citation_coverage':sum(coverage)/len(coverage),'answer_decision_accuracy':sum(abstention)/len(abstention)}
metrics=evaluate(cases); print(metrics)
assert round(metrics['retrieval_recall'],3)==0.667
assert round(metrics['citation_coverage'],3)==0.5


In [ ]:
gate={'retrieval_recall':0.8,'citation_coverage':0.9,'answer_decision_accuracy':0.95}
failures={name:(metrics[name],minimum) for name,minimum in gate.items() if metrics[name]<minimum}
print('release gate failures:',failures); assert failures


평균만 보지 말고 source 유형, 언어, 권한 집단과 freshness 구간별로 slice합니다. ACL 누출은 평균 metric이 아니라 허용치 0의 security invariant로 관리하세요.